In [1]:
from nilearn import datasets
import nibabel as nib
import numpy as np
from neuromaps import nulls, stats
from nilearn.maskers import NiftiLabelsMasker

# Load your wake and sleep group-level t-maps
wake_img = nib.load('/Users/zach/2025_RA/matthias/bids_sleepmreye/derivatives/mreyemove/group/func/glm_brain/task-sleep_contrast-mreyemove_W-clean-with-fd-clamped-original-sleep/second_level_flame/flameo/stats/tstat1.nii.gz')
sleep_img = nib.load('/Users/zach/2025_RA/matthias/bids_sleepmreye/derivatives/mreyemove/group/func/glm_brain/task-sleep_contrast-mreyemove_S-clean-with-fd-clamped-original-sleep/second_level_flame/flameo/stats/tstat1.nii.gz')

# Parcellate using Schaefer atlas (same one you used for FC)
parc = datasets.fetch_atlas_schaefer_2018(n_rois=500).maps
masker = NiftiLabelsMasker(labels_img=parc, standardize=False)
vals_wake = masker.fit_transform(wake_img).squeeze()   # (500,)
vals_sleep = masker.fit_transform(sleep_img).squeeze()

# Generate spatial-autocorrelation-preserving null maps for wake
null_maps = nulls.burt2020(
  vals_wake,
  atlas='mni152',
  density='2mm',
  parcellation=parc,
  n_perm=1000,
  seed=42,
  n_proc=4,
)

# Compare: observed r + p-value against null
corr, pval = stats.compare_images(vals_wake, vals_sleep, nulls=null_maps)
print(f'r = {corr:.3f}, p_spin = {pval:.4f}')


/Users/zach/.virtualenvs/testmreyemove/lib/python3.11/site-packages/neuromaps/datasets/utils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


[fetch_atlas_schaefer_2018] Dataset found in /Users/zach/nilearn_data/schaefer_2018
[fetch_files] Downloading data from https://files.osf.io/v1/resources/4mw3a/providers/osfstorage/60b684bd3a6df1021cd4e983 ...
Downloaded 4382720 of 21146249 bytes (20.7%%,    3.8s remaining)
[fetch_files]  ...done. (5 seconds, 0 min)

[fetch_files] Extracting data from /Users/zach/neuromaps-data/45ecf8cb43641c5d8ba168701738f583/MNI1522mm.tar.gz...
[fetch_files] .. done.


KeyboardInterrupt: 